In [1]:
import json
import os
from typing import List, Dict
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pandas as pd
from util import *

In [26]:
import json
import os
from typing import List, Dict
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


def ensure_nltk_data():
    """Ensure NLTK data is downloaded only if not already present"""
    try:
        # Try to access stopwords - will raise LookupError if not downloaded
        stopwords.words('english')
    except LookupError:
        nltk.download('stopwords')
        
    try:
        # Try tokenizing - will raise LookupError if punkt is not downloaded
        word_tokenize('test')
    except LookupError:
        nltk.download('punkt')

class ECHRCaseRetrieval:
    def __init__(self, cases_directory: str):
        """
        Initialize the retrieval system with a directory of JSON case files.
        
        Args:
            cases_directory (str): Path to directory containing JSON case files
        """
        self.cases = []
        self.case_facts = []
        self.data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed'
        self.bm25 = None
        
        # Download required NLTK data
        #nltk.download('punkt')
        #nltk.download('stopwords')
        ensure_nltk_data()
        self.stop_words = set(stopwords.words('english'))
        
        # Load all cases
        self._load_cases(cases_directory)
        # Create BM25 index
        self._create_index()
    
    def _load_cases(self, directory: str) -> None:
        """Load all JSON files from the specified directory."""
        for filename in os.listdir(self.data_path):
            if filename.endswith('.json'):
                file_path = os.path.join(directory, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        case = json.load(f)
                        if 'facts' in case:
                            self.cases.append(case['itemid'])
                            # Preprocess the facts text
                            tokenized_facts = self._preprocess_text(case['facts'])
                            self.case_facts.append(tokenized_facts)
                except Exception as e:
                    print(f"Error loading {filename}: {e}")
    
    def _preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text by tokenizing and removing stopwords.
        
        Args:
            text (str): Input text
            
        Returns:
            List[str]: List of preprocessed tokens
        """
        # Tokenize
        tokens = word_tokenize(text.lower())
        # Remove stopwords and non-alphabetic tokens
        tokens = [token for token in tokens if token.isalpha() and token not in self.stop_words]
        return tokens
    
    def _create_index(self) -> None:
        """Create BM25 index from preprocessed case facts."""
        self.bm25 = BM25Okapi(self.case_facts)
    
    def search_similar_cases(self, query_facts: str, top_k: int = 5) -> List[Dict]:
        """
        Search for similar cases based on facts.
        
        Args:
            query_facts (str): Facts text to search for
            top_k (int): Number of similar cases to return
            
        Returns:
            List[Dict]: List of top-k similar cases with scores
        """
        # Preprocess query
        query_tokens = self._preprocess_text(query_facts)
        
        # Get BM25 scores
        scores = self.bm25.get_scores(query_tokens)
        
        # Get top-k cases
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        
        # Prepare results
        results = []
        for idx in top_indices:
            results.append({
                'case': self.cases[idx],
                'score': scores[idx]/(len(self.case_facts[idx])**2) # Normalize by length of case facts
            })
        
        return results

In [27]:
data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed'
retrieval_system = ECHRCaseRetrieval(data_path)

In [30]:
list_1, list_2, list_3, list_4 = read_data()

In [31]:
query = load_json(os.path.join(data_path, list_1[0]))['facts']
similar_cases = retrieval_system.search_similar_cases(query, top_k=5)

print("\nTop 5 Similar Cases:")
similar_cases


Top 5 Similar Cases:


[{'case': '001-220960', 'score': 0.006907113311570533},
 {'case': '001-173623', 'score': 1.763674558577168e-05},
 {'case': '001-146372', 'score': 5.8120824921217265e-05},
 {'case': '001-95771', 'score': 7.310123645404215e-05},
 {'case': '001-108599', 'score': 2.556396600277659e-05}]

In [32]:
query = load_json(os.path.join(data_path, list_4[0]))['facts']
similar_cases = retrieval_system.search_similar_cases(query, top_k=5)

print("\nTop 5 Similar Cases:")
similar_cases


Top 5 Similar Cases:


[{'case': '001-229412', 'score': 0.12503816222256958},
 {'case': '001-214753', 'score': 0.09883837604879533},
 {'case': '001-222907', 'score': 0.10826710964233291},
 {'case': '001-220546', 'score': 0.11880133495531967},
 {'case': '001-228993', 'score': 0.11880133495531967}]

In [12]:
a= ['001-228672',
 '001-228677',
 '001-226426',
 '001-228680',
 '001-224571',
 '001-229406',
 '001-228676',
 '001-227738',
 '001-225889',
 '001-220544',
 '001-228995',
 '001-229412']


b = ['001-229412',
    '001-214753',
    '001-222907',
    '001-220546']


# intersection of two lists
c = list(set(a) & set(b))
print(c)


['001-229412']
